In [23]:
import pandas as pd
rating_movies = pd.read_csv('data/ratings_movies.csv', sep=',')

In [42]:
rating_movies.head()

,Unnamed: 0,userId,movieId,rating,date,title,genres,year_release
0,0,1,1,4.0,2000-07-30 18:45:03,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995
1,1,1,3,4.0,2000-07-30 18:20:47,Grumpier Old Men (1995),Comedy|Romance,1995
2,2,1,6,4.0,2000-07-30 18:37:04,Heat (1995),Action|Crime|Thriller,1995
3,3,1,47,5.0,2000-07-30 19:03:35,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,1995
4,4,1,50,5.0,2000-07-30 18:48:51,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,1995


In [24]:
#библиотека для регулярных выражений
import re 
def get_year_release(arg):
    #находим все слова по шаблону "(DDDD)"
    candidates = re.findall(r'\(\d{4}\)', arg) 
    # проверяем число вхождений
    if len(candidates) > 0:
        #если число вхождений больше 0,
	#очищаем строку от знаков "(" и ")"
        year = candidates[0].replace('(', '')
        year = year.replace(')', '')
        return int(year)
    else:
        #если год не указан, возвращаем None
        return None

In [25]:
rating_movies['year_release'] = rating_movies['title'].apply(get_year_release)
rating_movies['year_release'] = rating_movies['year_release'].astype('Int16') 

In [26]:
rating_movies['year_release']

0         1995
1         1995
2         1995
3         1995
4         1995
          ... 
100831    2017
100832    2017
100833    2017
100834    2017
100835    2017
Name: year_release, Length: 100836, dtype: Int16

In [27]:
rating_movies['year_release'].notnull().sum()

np.int64(100818)

In [28]:
mask1 = rating_movies['year_release'] == 1999

In [39]:
rating_movies[mask1].groupby(by='title').mean(numeric_only=True).loc[::, 'rating'].sort_values()

title
Bloodsport: The Dark Kumite (1999)                  0.5
Trippin' (1999)                                     1.0
Chill Factor (1999)                                 1.0
From Dusk Till Dawn 2: Texas Blood Money (1999)     1.0
Simon Sez (1999)                                    1.0
                                                   ... 
Mickey's Once Upon a Christmas (1999)               5.0
On the Ropes (1999)                                 5.0
Trailer Park Boys (1999)                            5.0
Five Senses, The (1999)                             5.0
George Carlin: You Are All Diseased (1999)          5.0
Name: rating, Length: 261, dtype: float64

In [43]:
mask1 = rating_movies['year_release'] == 2010
rating_movies[mask1].groupby(by='genres').mean(numeric_only=True).loc[::, 'rating'].sort_values()

genres
Action|Sci-Fi                        1.000000
Action|Adventure|Horror              1.500000
Action|Drama|Fantasy                 1.500000
Crime|Romance                        1.500000
Adventure|Comedy|Fantasy             1.833333
                                       ...   
Crime                                4.750000
Adventure|Children|Comedy|Mystery    5.000000
Animation|Children|Mystery           5.000000
Animation|Drama|Fantasy|Mystery      5.000000
Comedy|Musical                       5.000000
Name: rating, Length: 119, dtype: float64

In [47]:
rating_movies.groupby(by='userId').nunique().loc[::, 'genres'].sort_values(ascending=False)

userId
599    524
414    482
448    403
380    399
474    395
      ... 
494     15
578     15
214     13
85      13
245     13
Name: genres, Length: 610, dtype: int64

In [55]:
rating_movies.groupby(by='userId')['rating'].agg(['count', 'mean']).sort_values(by=['count', 'mean'], ascending=[True, False])

,count,mean
userId,,
53,20,5.000000
595,20,4.200000
189,20,4.100000
569,20,4.000000
278,20,3.875000
...,...,...
274,1346,3.235884
448,1864,2.847371
474,2108,3.398956


In [72]:
mask1 = rating_movies['year_release'] == 2018

In [73]:
pivot = rating_movies[mask1].pivot_table(
    values='rating',
    index='genres',
    aggfunc=['count','mean']
)
pivot = pivot[pivot[('count', 'rating')] > 10]
pivot.sort_values(by=[('mean', 'rating')], ascending=False)

,count,mean
,rating,rating
genres,,
Action|Adventure|Sci-Fi,14,3.928571
Action|Comedy|Sci-Fi,12,3.875000


In [77]:
rating_movies['year_rating'] = pd.to_datetime(rating_movies['date']).dt.year
pivot = rating_movies.pivot_table(
    values='rating',
    index=['year_rating', 'genres'],
    aggfunc='mean'
)
pivot

rating
year_rating genres                                                   
1996        Action                                           2.730769
            Action|Adventure                                 3.454545
            Action|Adventure|Animation|Horror|Sci-Fi         3.666667
            Action|Adventure|Children|Comedy|Fantasy|Sci-Fi  2.250000
            Action|Adventure|Comedy|Crime                    3.287500
...                                                               ...
2018        Sci-Fi                                           3.128571
            Sci-Fi|IMAX                                      3.500000
            Sci-Fi|Thriller                                  3.562500
            Thriller                                         3.296296
            Western                                          3.611111

[9273 rows x 1 columns]

In [83]:
pivot.xs('Comedy', level='genres')

,rating
year_rating,
1996,3.228571
1997,3.409091
1998,3.000000
1999,3.606061
2000,3.141291
2001,3.318408
2002,3.198556
2003,3.120066
2004,3.356877
